In [1]:
#!pip install tiktoken --q

In [2]:
import tiktoken

In [3]:
MODEL_ID = "gpt-4o-mini"

In [7]:
def num_tokens_from_messages(messages):
    """
    Simplified token counter specifically for gpt-4o-mini.
    """
    encoding = tiktoken.encoding_for_model(MODEL_ID)
    
    # Standard chat markup overhead for modern OpenAI chat models
    tokens_per_message = 3
    tokens_per_name = 1

    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(str(value)))
            if key == "name":
                num_tokens += tokens_per_name
                
    num_tokens += 3  # every reply is primed with assistant output
    return num_tokens

In [4]:
def run_query(query: str):
    # 1. Choose a model encoding
    # 'gpt-4o-mini' uses the 'cl100k_base' encoding
    print(f"--- Loading encoding for model: {MODEL_ID} ---")
    
    try:
        encoding = tiktoken.encoding_for_model(MODEL_ID)
    except KeyError:
        print(f"Model {model_name} not found, falling back to cl100k_base.")
        encoding = tiktoken.get_encoding("cl100k_base")

    # 2. Query Text
    print(f"\nOriginal Text:\n{query}")

    # 3. Encode text into token IDs
    token_ids = encoding.encode(query)
    print(f"\nToken IDs ({len(token_ids)} tokens total):")
    print(token_ids)

    # 4. Decode token IDs back to text
    decoded_text = encoding.decode(token_ids)
    print(f"\nDecoded Text:\n{decoded_text}")

    # 5. Inspect individual tokens
    print("\n--- Inspecting Individual Tokens ---")
    for token_id in token_ids:
        single_token_bytes = encoding.decode_bytes([token_id])
        print(f"ID: {token_id:5d} | Bytes: {repr(single_token_bytes)}")

    # 6. Counting tokens for Chat Messages
    print("\n--- Chat Message Token Counting ---")
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": query}
    ]
    
    total_tokens = num_tokens_from_messages(messages)
    print(f"Total tokens for the chat payload ({MODEL_ID}): {total_tokens}")

In [5]:
# Source:
#The Book of Nonsense
#(Written by Edward Lear, the father of the modern limerick, in 1846)

sample_string = """There was an Old Man with a beard,
Who said, It is just as I feared!—
Two Owls and a Hen,
Four Larks and a Wren,
Have all built their nests in my beard!
"""

In [8]:
run_query(sample_string)

--- Loading encoding for model: gpt-4o-mini ---

Original Text:
There was an Old Man with a beard,
Who said, It is just as I feared!—
Two Owls and a Hen,
Four Larks and a Wren,
Have all built their nests in my beard!


Token IDs (44 tokens total):
[5632, 673, 448, 14583, 3265, 483, 261, 95513, 412, 20600, 2059, 11, 1225, 382, 1327, 472, 357, 90672, 0, 123101, 13866, 63465, 3973, 326, 261, 18072, 412, 35604, 451, 13455, 326, 261, 486, 1687, 412, 15334, 722, 8113, 1043, 174186, 306, 922, 95513, 4175]

Decoded Text:
There was an Old Man with a beard,
Who said, It is just as I feared!—
Two Owls and a Hen,
Four Larks and a Wren,
Have all built their nests in my beard!


--- Inspecting Individual Tokens ---
ID:  5632 | Bytes: b'There'
ID:   673 | Bytes: b' was'
ID:   448 | Bytes: b' an'
ID: 14583 | Bytes: b' Old'
ID:  3265 | Bytes: b' Man'
ID:   483 | Bytes: b' with'
ID:   261 | Bytes: b' a'
ID: 95513 | Bytes: b' beard'
ID:   412 | Bytes: b',\n'
ID: 20600 | Bytes: b'Who'
ID:  2059 | Bytes: b

## **61 tokens demystified**

To understand how  **44-token poem** turns into a **61-token total**, it helps to look at how the OpenAI API reads chat messages.

When you send text to a chat model, the API doesn't just look at your raw words. It wraps everything in hidden formatting tags so the model knows who is speaking (the system, the user, or the assistant).

Here is the exact step-by-step bridge from 44 to 61:

1. **Raw Poem (44 tokens):**
This is the pure text of the limerick that your script printed out (`Token IDs (44 tokens total)`).
2. **User Message Formatting (+3 tokens):**
Your script's function adds a little extra structural weight to your poem—accounting for the `"role": "user"` tag and the API's internal message boundaries.
*(44 poem tokens + 3 structural tokens = **47 tokens** for the user message block).*
3. **System Message (11 tokens):**
Your instructions (`"You are a helpful assistant."`) plus its own structural formatting tags.
4. **Assistant Priming (+3 tokens):**
The script adds 3 final tokens at the very end (like `<|im_start|>assistant`) to act as a prompt cue, telling the model it's time to generate its reply.

### The Final Equation:

$$\mathbf{47} \text{ (User message + poem)} + \mathbf{11} \text{ (System message)} + \mathbf{3} \text{ (Priming cues)} = \mathbf{61 \text{ total tokens}}$$

In short: Your poem takes up 44 tokens, and the extra 17 tokens come from the system instructions, roles, and formatting required to send a chat message to the API!